## <img style="float: left; padding-right: 10px; width: 45px" src="ExtensionFlag.jpg"> CSCI-E104: Advanced Deep Learning, Spring 2026

### Lab 7: Text-to-Speech (TTS) Transformers
**Harvard University Extension School - Prof. Zoran B. Djordjević, Blagoje Djordjević**<br/>
**Joan Imrich  13-March-2026**<br/>
<hr style="height:2pt">

<a class="toc" id="toc"></a>

- **TTS-Phonemizer Demo:** [Festival, eSpeak)](#phon-demo)
- **TTS Transformer Pipelines:** [Models, APIs, Libraries, Environments](#tts-env)
- **STT-TTS Demo:** [(Whisper) STT transcript → (TTS Tacotron2 + WaveGlow)](#stt-tts-demo)

In [1]:
!sudo apt-get update && sudo apt-get install -y espeak-ng libespeak-ng-dev festival festival-dev
!sudo apt-get install -y festvox-kallpc16k

# 2. Install a default voice (Festival often fails without one)
!sudo apt-get install -y festvox-kallpc16k
!pip install espeak-ng-python


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,914 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InR

In [2]:
!pip install whisper phonemizer pyfestival

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 40.1 MB/s eta 0:00:00
  Created wheel for whisper: filename=whisper-1.1.10-py3-none-any.whl size=41120 sha256=ba1b5b32bbf55f24fe6025ff90322efdfda7abbd387687fb4040617003548c96
  Stored in directory: /root/.cache/pip/wheels/34/b8/4e/9c4c3351d670e06746a340fb4b7d854c76517eec225e5b32b1
  Created wheel for pyfestival: filename=pyfestival-0.6-cp312-cp312-linux_x86_64.whl size=682901 sha256=0565dc9fcd4aef1bb775addf7be27df2c4326b5392f2d5338167f6d2b7780ce1
  Stored 

## <a class="phomemizer-demo" id="phomemizer-demo"> ((((((((((((( TTS - Phonemizer Demo ))))))))))) </a>

[Go back to Notebook Content](#toc)


**Phonemes (sounds), use an IPA‑like alphabet. It shows how the text‑to‑speech system will *pronounce* what you wrote, not how it’s spelled**

- IPA (International Phonetic Alphabet) is a standard symbolic set for representing speech sounds

<img style="float: right; padding-left: 10px;" width="500" src="articulatory_coding.png">

- `həloʊ` ≈ “hello”  
- `wʌt` ≈ “what”  
- `ɪz` ≈ “is”  
- `ðə` ≈ “the” (the *th* in “this”)  
- `dɪfɹəns` ≈ “difference”  
- `bᵻtwiːn` ≈ “between”  
- `ɐ kɛmɪst` ≈ “a chemist”  
- `ænd` ≈ “and”  
- `plʌmɚ` ≈ American “plumber”

Second and third lines have Markdown/formatting in them (like `**unionized**), and the phonemizer is trying to *pronounce* those control codes as if they were words:
- `æstɚɹɪskɐstɚɹɪsk` ≈ “asterisk asterisk” (from `**`) is turned into pronounceable sounds too.

## TTS pipeline example
Festival, eSpeak, and IPA all sit in the TTS “front end” / “text‑to‑phone” space, but at different levels

- - IPA is the **symbolic alphabet** you often target for the phoneme layer that both **Festival and eSpeak** can approximate or output.

A common modern pipeline using these components looks like:  
1. **Text → phones**  
   - Use Festival or eSpeak (directly, or via `phonemizer`) to convert text into phoneme sequences, often in IPA or an IPA‑like phone set.  
2. **Phones (+ prosody) → acoustics**  
   - Feed those phonemes into Tacotron2, FastSpeech, etc., which predict mel‑spectrograms or other acoustic features.  
3. **Acoustics → waveform**  
   - Use a vocoder (WaveGlow, HiFi‑GAN, etc.) to synthesize the final waveform.


## IPA phonemes

IPA (International Phonetic Alphabet) is a **standard symbolic set** for representing speech sounds:  
- Each symbol corresponds to a specific **phone** (articulatory definition: place/manner of articulation, voicing, etc.).  
- It is **language‑independent**: English /p/ and Spanish /p/ use the same basic IPA symbol, even if detailed implementation differs.  
- In TTS, IPA can be used as the **internal phoneme inventory**: the front‑end (Festival, eSpeak, or custom grapheme‑to‑phoneme) converts text to IPA sequences, and the acoustic model learns to map IPA sequences to acoustic features.  


### **Important** TTS Config Checklist:
**<font color="#DC143C">Pay attention to Sound File Format, Online APIs require internet vs Offline packages, Conda ENV / Kernel (activate / restart)**</font>
- **Match rates**: Ensure audio files use a sample rate supported by the playback device (e.g., 44.1kHz for CD audio, 48kHz for video).  
- **Avoid dynamic changes**:  <font color="#DC143C">Use fixed sample rates</font> for all playback to prevent distortion  
- **Check hardware limits**: Devices often have fixed or limited rate support (e.g., 16-bit/44.1kHz vs. 24-bit/192kHz)

For persistent issues, use **upsampling** (e.g., 768kHz) to bypass device limitations, though this may not be ideal for all use cases

In [3]:
import sys
import os
print(sys.executable)
print(sys.version)
# !{sys.executable} -m pip install phonemizer

/usr/bin/python3
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [4]:
import sys
import os
print(sys.executable)
print(sys.version)

#import coqui_tts
# print(coqui_tts.__version__)
#import coqui_tts
#from coqui_tts import TTS
#The API changed; you should not import TTS.api anymore.
#from TTS.api import TTS  # Covers XTTS-v2 and Coqui/TTS
import whisper
# from gtts import gTTS
# import gTTS
# !{sys.executable} -m pip install phonemizer
import phonemizer
from phonemizer import phonemize
from IPython.display import Markdown, display
import IPython.display as ipd  # For notebook audio playback
import torch
import torchaudio
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import nltk
print("All imports work!")
print(f"phonemizer: {phonemizer.__version__}")
#print(f"gTTS: {gtts.__version__}")
# print(f"whisper: {whisper.__version__}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"nltk: {nltk.__version__}")
print(f"torch: {torch.__version__}")
print(f"torchaudio: {torchaudio.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


/usr/bin/python3
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
All imports work!
phonemizer: 3.3.0
numpy: 2.0.2
pandas: 2.2.2
scikit-learn: 1.6.1
nltk: 3.9.1
torch: 2.10.0+cu128
torchaudio: 2.10.0+cu128
Using device: cuda


In [ ]:

from phonemizer import phonemize
from IPython.display import Markdown, display

# Text to be phonemized
text = """Hello, what is the difference between a chemist and plumber?
A chemist experiments with **unionized** molecule,
A plumber performs **unionized** work"""

display(Markdown(text))

# Phonemize the text using eSpeak backen
phonemes = phonemize(
    text,
    language='en-us',
    backend='espeak',
    preserve_punctuation=False,
    strip=True
)

#print("\nOriginal text:", text)
print("\nPhonemized text:", phonemes)
# sentence = "Hello, what is the difference between a chemist and plumber? A chemist experiments with an un ionized molecule, A plumber performs unionized work"
sentence = "Hi, this is a student testing the e104 text to phenomes, Does it work or do I have to be worried?"
phonetic_output = phonemize(sentence, language='en-us', backend='festival', strip=True)

print("\nPhonetic Transcription (Festival Backend):\n", phonetic_output)

# Generate speech
import subprocess

output_wav_file = "espeak_output_unionized.wav"
subprocess.run(["espeak-ng", "-v", "en-us", "-w", output_wav_file, sentence])

print(f"Speech generated and saved as {output_wav_file}")

# Play the audio in the notebook
from IPython.display import Audio, display
display(Audio(output_wav_file))

## Printed out below is the phonetic transcription of the input sentence, which shows how the words are pronounced using the Festival backend. 

Hello, what is the difference between a chemist and plumber?
A chemist experiments with **unionized** molecule,
A plumber performs **unionized** work


Phonemized text: həloʊ wʌt ɪz ðə dɪfɹəns bᵻtwiːn ɐ kɛmɪst ænd plʌmɚ
ɐ kɛmɪst ɛkspɛɹɪmənts wɪð æstɚɹɪskɐstɚɹɪsk juːniənaɪzd æstɚɹɪskɐstɚɹɪsk mɑːlɪkjuːl
ɐ plʌmɚ pɚfɔːɹmz æstɚɹɪskɐstɚɹɪsk juːniənaɪzd æstɚɹɪskɐstɚɹɪsk wɜːk

Phonetic Transcription (Festival Backend):
 hhay dhaxs ihz ax stuwdaxnt tehstaxng dhax iy wahn zihrow faor tehkst tax faxnowmz dahz iht werk aor duw ay hhaev tax biy weriyd
Speech generated and saved as espeak_output_unionized.wav


## **Festival is a full TTS system** from CSTR (Edinburgh)

####  festival provides a rich internal structure and pluggable waveform backends

- When people say **Festival backend** in practice they often mean “use Festival to do the front‑end (text→phones, prosody, utterance) and then hand off phones + prosody to another synthesizer such as MBROLA or a neural vocoder.
- A **modular architecture**: separate modules for tokenization, text normalization, linguistic analysis (POS, phrasing), letter‑to‑sound, prosody, and waveform generation. Voices (diphone, unit‑selection, etc.) plug into this pipeline as separate data + Scheme code
- A core **utterance structure**: an internal graph‑like representation with items (words, syllables, segments) and relations connecting them; each synthesis module reads and writes to this structure
- Multiple **waveform backends**: can synthesize directly (formant, diphone, unit selection) or delegate to external engines such as MBROLA, using Festival only for the linguistic front‑end.

<a class="phone-demo" id="phon-demo"></a>
## <font color="#DC143C"> Phonemizers: eSpeak, Festival</font>

[Go back to Notebook Content](#toc)

**This demo may be helpful for  <font color="#DC143C"> HW assingment 7</font>**


In [6]:
# Run Phonemizer on an example sentence
from phonemizer import phonemize

sentence = "Hi, this is a student testing the e104 text to phenomes, Does it work or do I have to be worried?"
phonetic_output = phonemize(sentence, language='en-us', backend='festival', strip=True)

print("Phonetic Transcription (Festival Backend):\n\n", phonetic_output)


Phonetic Transcription (Festival Backend):

 hhay dhaxs ihz ax stuwdaxnt tehstaxng dhax iy wahn zihrow faor tehkst tax faxnowmz dahz iht werk aor duw ay hhaev tax biy weriyd


## Question 2

### Usng espeak backend to generate the audio file

## **eSpeak lightweight TTS/phoneme generator**

#### eSpeak is a compact, rule‑based TTS engine, often used only for its grapheme‑to‑phoneme rules
- In tools like **`phonemizer`** , selecting the “espeak” backend means “call eSpeak to convert text to phonemes” without necessarily using eSpeak’s audio at all.
- It includes its own **text analysis and letter‑to‑sound rules**, implemented mainly as hand‑crafted phoneme rules for many languages.  
- It outputs **phoneme sequences** with stress markers and basic prosodic cues; these phonemes can be rendered by eSpeak’s own synthesizer or used as input to another backend (for example, phoneme‑level neural TTS).  
- It focuses on **small footprint and coverage** rather than super‑natural audio quality, which is why it’s still used inside other systems as a phoneme/feature generator rather than as the final waveform engine.  




In [7]:
# Generate speech
import subprocess

output_wav_file = "espeak_phoneme_output.wav"
subprocess.run(["espeak-ng", "-v", "en-us", "-s", "133",  "-w", output_wav_file, sentence])
    # "espeak",
    # "-v", "en-us+f2",   # voice, US English, female variant
    # "-s", "200",        # speed 200 wpm
    # "-p", "60",         # slightly higher pitch
    # "-a", "120",        # a bit louder
    # "-w", output_wav_file,

print(f"Speech generated and saved as {output_wav_file}")

# Play the audio in the notebook
from IPython.display import Audio, display
display(Audio(output_wav_file))


Speech generated and saved as espeak_phoneme_output.wav


In [13]:
# Run Phonemizer on an example sentence
from phonemizer import phonemize

sentence = "This is an assignment 7 validation test."
phonetic_output = phonemize(sentence, language='en-us', backend='espeak', strip=True)

print("Phonetic Transcription (eSpeak Backend):\n\n", phonetic_output)


Phonetic Transcription (eSpeak Backend):

 ðɪs ɪz ɐn ɐsaɪnmənt sɛvən vælɪdeɪʃən tɛst


In [14]:
# Generate speech
import subprocess

output_wav_file_2 = "espeak_esound_output.wav"
subprocess.run(["espeak-ng", "-v", "en-us", "-w", output_wav_file_2, sentence])

print(f"Speech generated and saved as {output_wav_file_2}")
# Play the audio in the notebook
from IPython.display import Audio, display
display(Audio(output_wav_file_2))


Speech generated and saved as espeak_esound_output.wav


## <a class="tts-env" id="tts-env"> ((((((((((((( TTS - Pipelines ))))))))))) </a>

[Go back to Notebook Content](#toc)

## <font color="#DC143C"> TTS Transformer Pipelines</font>

**This demo may be helpful for  <font color="#DC143C"> HW assingment 7</font>**
[see HW7 - Coqui Repo](https://github.com/coqui-ai/TTS), [Source Paper](https://arxiv.org/pdf/2006.04558)

### How this fits together in a TTS pipeline
**In short: **Tacotron uses attention** as a single alignment block around an RNN decoder, **XTTS uses transformer attention** as the primary modeling tool across the whole network, and **HiFi‑GAN mainly relies on convolutions, assuming the attention/alignment work is already done upstream.**

- Attention and transformers sit at the **core** of XTTS‑style models, while Tacotron and HiFi‑GAN use attention and self‑attention in more specialized ways in the overall TTS stack.

- **TTS (inpput: text transcript) → Speech translation (output: wav, mp3, mp4 files)**
-  `String Characters → Attention RNN → Mel Spectrogram → GAN Waveform → Sound `
-  **Text Strings → Tacotron2-DDC → Mel-spectrogram (80 mel bins) → HiFiGAN Waveform → WAV Audio (22kHz) Speech**

## How this fits together in a TTS pipeline

- **Tacotron + HiFi‑GAN**:  
  - Tacotron: encoder–decoder with attention → produces mel‑spectrograms with learned text–audio alignment.  
  - HiFi‑GAN: convolutional vocoder → converts mels to waveform, no text attention.  

- **XTTS + HiFi‑GAN‑class vocoder**:  
  - XTTS: transformer encoder+decoder with self‑attention and cross‑attention for both text understanding and text–audio alignment, plus voice‑cloning conditioning.  
  - Vocoder (often HiFi‑GAN‑like): converts XTTS’s acoustic representation to waveform.


## Tacotron: classic attention encoder–decoder

In Tacotron/Tacotron2, attention is the main mechanism that aligns text and acoustic frames.

- The encoder turns text (characters or phonemes) into a sequence of hidden embeddings.  
- The decoder is an autoregressive RNN that predicts mel‑spectrogram frames one step at a time.  
- At each decoding step, an attention module computes weights over encoder states so the decoder “looks at” the right text positions (roughly left‑to‑right, monotonic) while generating each frame.  
- This attention alignment is what lets Tacotron learn durations and rhythm implicitly, instead of needing precomputed alignments.

Transformers in classic Tacotron are limited: the original versions use CNN + RNN + attention, not full transformer stacks; the attention itself is still a learned, content‑ and location‑based mechanism.

## XTTS: transformer-style attention everywhere

XTTS/XTTS‑v2 are much more transformer‑like.

- Text is encoded with transformer layers (self‑attention blocks) to capture long‑range dependencies and context across the whole sentence or paragraph.  
- The acoustic or “decoder” side uses cross‑attention to map from text representations to audio tokens or intermediate acoustic features, often in a GPT‑like autoregressive or semi‑autoregressive fashion.  
- Self‑attention layers also model dependencies within the generated audio sequence (e.g., ensuring consistent prosody and style across an utterance).  
- Because transformers can handle long contexts, XTTS can better keep voice, emotion, and phrasing consistent over long passages (audiobooks, dialogues) compared to a simple RNN+attention Tacotron.

So in XTTS, attention is not just an alignment tool; it is the main building block (multi‑head self‑attention and cross‑attention) of the model’s architecture.

## HiFi‑GAN: convolutional vocoder with limited/no alignment attention

**HiFi‑GAN itself is a neural vocoder: it turns mel‑spectrograms (or similar features) into waveforms.**

- The core HiFi‑GAN generator is convolutional and does not rely on text–audio attention; alignment has already been handled upstream by Tacotron/XTTS or another acoustic model.  
- It may use internal mechanisms like multi‑receptive‑field convolutions to capture local and wider temporal context, but not encoder–decoder text attention.  
- In modern pipelines, the “attention + transformer” work is done before the vocoder; HiFi‑GAN focuses on high‑fidelity, fast waveform generation given a conditioned feature sequence.





## <a class="stt-tts-demo" id="stt-tts-demo"> (((((((((((((  Whisper STT → TTS pipeline  ))))))))))) </a>

[Go back to Notebook](#toc)


## <font color="#DC143C"> STT → TTS pipeline  (local)</font>

We can extend  **Whisper STT** script into a full **STT→TTS pipeline** by adding:
-  **Tacotron2 + WaveGlow (or other vocoder) stage that takes text and produces a waveform on CUDA**
- Below is a compact example wired for this environment (torch 2.1.2+cu121, torchaudio 2.1.2+cu121, phonemizer 3.3.0).

Optional for STT-TTS pipeline: **use phonemizer**
- If you want to **explicitly phonemize your text before Tacotron2** (sometimes improves pronunciation consistency)
- For this to work well, you may need to adapt the Tacotron2 symbol set to match the phoneme alphabet; the pretrained NVIDIA models are usually trained on character inputs, so start with raw text and only move to phonemes if you’re comfortable tweaking symbol tables.


In [12]:
from phonemizer import phonemize

text = "With the noise in Auto Encoders, we will have an image and perfect image, \
    and then they will add noise to that image. And then we will push that noise, image with noise, \
    through Auto Encoder. Auto Encoder and Coder of Auto Encoder will create a Latin space, \
    and because Latin space is small, small dimensionally, \
    it will remember only important feature on the image, and it will basically ignore the noise. \ And then we will expand that Latin space image into a real-sized image. And what will come out \
    will be a new image, and we will compare that new image with the original image without noise. \
    So in that way, Auto Encoder will learn to remove noise, because it will be asked \
    to compare noise image or to transform noise image into perfect image."


phoneme_text = phonemize(
    text,
    language="en-us",
    backend="espeak",
    strip=True,
    njobs=1
)
print("Phonemes:\n\n", phoneme_text)
# Then pass `phoneme_text` to utils.prepare_input_sequence instead of `text`

<>:7: SyntaxWarning: invalid escape sequence '\ '
<>:7: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_3727/78102749.py:7: SyntaxWarning: invalid escape sequence '\ '
  it will remember only important feature on the image, and it will basically ignore the noise. \ And then we will expand that Latin space image into a real-sized image. And what will come out \


Phonemes:

 wɪððə nɔɪz ɪn ɔːɾoʊ ɛŋkoʊdɚz wiː wɪl hæv ɐn ɪmɪdʒ ænd pɜːfɛkt ɪmɪdʒ ænd ðɛn ðeɪ wɪl æd nɔɪz tə ðæt ɪmɪdʒ ænd ðɛn wiː wɪl pʊʃ ðæt nɔɪz ɪmɪdʒ wɪð nɔɪz θɹuː ɔːɾoʊ ɛŋkoʊdɚɹ ɔːɾoʊ ɛŋkoʊdɚ ænd koʊdɚɹ ʌv ɔːɾoʊ ɛŋkoʊdɚ wɪl kɹiːeɪt ɐ lætɪn speɪs ænd bɪkʌz lætɪn speɪs ɪz smɔːl smɔːl dᵻmɛnʃənəli ɪt wɪl ɹᵻmɛmbɚɹ oʊnli ɪmpoːɹtənt fiːtʃɚɹ ɔnðɪ ɪmɪdʒ ænd ɪt wɪl beɪsɪkli ɪɡnoːɹ ðə nɔɪz bækslæʃ ænd ðɛn wiː wɪl ɛkspænd ðæt lætɪn speɪs ɪmɪdʒ ɪntʊ ɐ ɹiːəlsaɪzd ɪmɪdʒ ænd wʌt wɪl kʌm aʊt wɪl biː ɐ nuː ɪmɪdʒ ænd wiː wɪl kəmpɛɹ ðæt nuː ɪmɪdʒ wɪððɪ ɚɹɪdʒɪnəl ɪmɪdʒ wɪðaʊt nɔɪz soʊ ɪn ðæt weɪ ɔːɾoʊ ɛŋkoʊdɚ wɪl lɜːn tə ɹᵻmuːv nɔɪz bɪkʌz ɪt wɪl biː æskt tə kəmpɛɹ nɔɪz ɪmɪdʒ ɔːɹ tə tɹænsfɔːɹm nɔɪz ɪmɪdʒ ɪntʊ pɜːfɛkt ɪmɪdʒ


## <font color="#DC143C"> STT → TTS pipeline (local)</font>
**load audio → STT → tokenize/print → TTS → save output WAV**
- Pipeline loads an input WAV, transcribes it to text via Whisper (STT),
- Prints tokenized sentences, then synthesizes echoed audio via Tacotron2+WaveGlow (TTS)
- Consider alternatives like **torch-audiokit or TTS library** for easier setup **(see TTS-video demo)**
- **TTS Implementation: Tacotron2+WaveGlow** require pretrained weights and custom utils (e.g., from NVIDIA's repo).
- **Dependencies:** Install **pip install torch transformers librosa nltk**
- **Performance:** Use **torch.no_grad() in inference**; add try/except for file I/O.
- **Customization:** Swap text_for_tts = transcript for summarization (e.g., via pipeline("summarization")).

Code uses Whisper locally, not via the OpenAI (or Azure OpenAI) hosted API.

Why it’s local
You’re calling the Hugging Face Transformers pipeline with a model ID like "openai/whisper-base":

In [11]:
import librosa
import torch
from transformers import pipeline
import soundfile as sf
import nltk

import whisper  # Original OpenAI Whisper, not transformers

# Downloads once, then works offline forever
model = whisper.load_model("base")  # ~74MB, saves to ~/.cache/whisper/


# One‑time NLTK data download (safe to call repeatedly)
nltk.download("punkt", quiet=True)
from nltk.tokenize import sent_tokenize

# Device selection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

###############################
# STT helpers (Whisper)
###############################

def load_whisper_model(model_name="base"):
    """Load Whisper ASR model via transformers pipeline."""
    return pipeline(
        "automatic-speech-recognition",
        model=f"openai/whisper-{model_name}",
        device=0 if DEVICE == "cuda" else -1,
    )

def transcribe_audio(model, audio_path: str) -> str:
    """Transcribe an audio file and return text."""
    result = model(audio_path)
    # Some pipelines return dict, some a list; handle both
    if isinstance(result, list):
        result = result[0]
    return result.get("text", "")

###############################
# TTS helpers (Coqui TTS)
###############################

from TTS.api import TTS

# You can choose a different pre‑trained model from Coqui’s docs if you like
TTS_MODEL_NAME = "tts_models/en/ljspeech/tacotron2-DDC"

_tts_model = None

def load_tts_model():
    """Lazy‑load a single‑speaker TTS model."""
    global _tts_model
    if _tts_model is None:
        print(f"Loading TTS model: {TTS_MODEL_NAME} ...")
        _tts_model = TTS(model_name=TTS_MODEL_NAME, progress_bar=False, gpu=DEVICE=="cuda")
    return _tts_model

def synthesize_tts_to_wav(text: str, out_path: str) -> str:
    """Synthesize text to a WAV file."""
    tts_model = load_tts_model()
    # Coqui TTS can write directly to file
    tts_model.tts_to_file(text=text, file_path=out_path)
    return out_path

###############################
# End-to-end pipeline
###############################

def stt_then_tts(
    in_wav: str,
    out_wav: str = "tts_response.wav",
    whisper_model_name: str = "base",
):
    # STT phase
    print(f"Loading Whisper ({whisper_model_name}) on {DEVICE}...")
    stt_model = load_whisper_model(whisper_model_name)

    print("Transcribing input audio...")
    transcript = transcribe_audio(stt_model, in_wav).strip()

    if not transcript:
        raise RuntimeError("STT returned empty transcript; aborting TTS.")

    print("STT transcript:")
    for sent in sent_tokenize(transcript):
        print("  ", sent)

    # TTS phase
    print("Synthesizing TTS from transcript...")
    # Option A: echo full transcript
    text_for_tts = transcript
    # Option B: only first sentence
    # sents = sent_tokenize(transcript)
    # text_for_tts = sents[0] if sents else transcript

    out_path = synthesize_tts_to_wav(text_for_tts, out_wav)
    print(f"TTS audio saved to: {out_path}")
    return out_path

if __name__ == "__main__":
    input_wav = "ZD_AE.wav"

    # Optional: verify it loads fine
    y, sr = librosa.load(input_wav, sr=None, mono=True)
    print(f"Loaded {input_wav} at {sr} Hz")

    stt_then_tts(input_wav, out_wav="ZD_AE_tts.wav")


AttributeError: module 'whisper' has no attribute 'load_model'

In [ ]:
# Environment (versions you gave):
# phonemizer==3.3.0
# whisper==20250625
# numpy==1.26.4
# pandas==1.5.3
# scikit-learn==1.8.0
# nltk==3.9.3
# torch==2.1.2+cu121
# torchaudio==2.1.2+cu121

import os
import torch
import torchaudio
import whisper
import librosa
import nltk
from nltk.tokenize import sent_tokenize

#nltk.download("punkt")
#DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
###############################
# TTS with Tacotron2 + WaveGlow
###############################

def load_tacotron2_waveglow():
    # Load Tacotron2 (text -> mel)
    tacotron2 = torch.hub.load(
        "NVIDIA/DeepLearningExamples:torchhub",
        "nvidia_tacotron2",
        model_math="fp32"
    ).to(DEVICE).eval()

    # Load WaveGlow (mel -> audio)
    waveglow = torch.hub.load(
        "NVIDIA/DeepLearningExamples:torchhub",
        "nvidia_waveglow",
        model_math="fp32"
    )
    waveglow = waveglow.remove_weightnorm(waveglow)
    waveglow = waveglow.to(DEVICE).eval()

    # TTS utilities (text normalization, sequence prep)
    utils = torch.hub.load(
        "NVIDIA/DeepLearningExamples:torchhub",
        "nvidia_tts_utils"
    )

    return tacotron2, waveglow, utils

@torch.no_grad()
def tts_tacotron2_waveglow(
    text: str,
    tacotron2,
    waveglow,
    utils,
    out_path: str = "tts_out.wav",
    sigma: float = 0.7
) -> str:
    # Prepare input sequence (handles text -> token IDs)
    sequences, lengths = utils.prepare_input_sequence([text])

    sequences = sequences.to(DEVICE)
    lengths = lengths.to(DEVICE)


    # 1) Tacotron2: text -> mel spectrogram
    mel, _, _ = tacotron2.infer(sequences, lengths)

    # 2) WaveGlow: mel -> audio waveform
    audio = waveglow.infer(mel, sigma=sigma)
    audio = audio[0].cpu()

    # 22050 Hz is the standard sampling rate for these pretrained models
    sr = 22050
    torchaudio.save(out_path, audio.unsqueeze(0), sample_rate=sr)

    return out_path


In [ ]:
###############################
# End-to-end pipeline
###############################

def stt_then_tts(
    in_wav: str,
    out_wav: str = "tts_response.wav",
    whisper_model_name: str = "base"
):
    # STT phase
    print(f"Loading Whisper ({whisper_model_name}) on {DEVICE}...")
    stt_model = load_whisper_model(whisper_model_name)
    print("Transcribing input audio...")
    transcript = transcribe_audio(stt_model, in_wav)
    print("STT transcript:")
    for sent in sent_tokenize(transcript):
        print("  ", sent)

    # TTS phase
    print("Loading Tacotron2 + WaveGlow on", DEVICE, "...")
    tacotron2, waveglow, utils = load_tacotron2_waveglow()

    print("Synthesizing TTS from transcript...")
    # Example: you can echo the full transcript or just a summary/first sentence
    text_for_tts = transcript  # or sent_tokenize(transcript)[0]
    out_path = tts_tacotron2_waveglow(
        text_for_tts,
        tacotron2,
        waveglow,
        utils,
        out_path=out_wav
    )
    print(f"TTS audio saved to: {out_path}")

if __name__ == "__main__":
    # Your original input file
    input_wav = "ZD_AE.wav"

    # Optional: verify it loads fine
    y, sr = librosa.load(input_wav, sr=None, mono=True)
    print(f"Loaded {input_wav} at {sr} Hz")

    stt_then_tts(input_wav, out_wav="ZD_AE_tts.wav")


##  Voice Synthesizing TTS from transcript (Whisper)

In [ ]:
from IPython.display import Audio

Audio("ZD_AE_tts.wav")  # Synthetic Cloned Voice


## Compare to Original Audio Speaker Clip

In [ ]:
Audio("./voices/zd/1.wav")  # Short Audio clip Human Voice

In [ ]:
import whisper
import os
import librosa
import librosa.display
print(whisper.__version__)
#!pip list
model = whisper.load_model("base")
print("Whisper model loaded successfully!")
# Load the small English model
#model = whisper.load_model("small")
test00_tts = "ZD_AE.wav" #mono_file2 = "ZD_AE.wav"
mono_signal_tts, sample_rate = librosa.load(test00_tts) #ZD_AE

# Transcribe ZD file
result = model.transcribe(test00_tts)
print("STT Transcription of ZD audiofile:")
for sent in sent_tokenize(result['text']):
  print("STT Transcript:: ", sent)

In [ ]:
from IPython.display import Audio
Audio("ZD_AE.wav", rate=22050)  # For Jupyter notebooks

### <font color="#DC143C">Chroma features</font>

- **Describe how energy is distributed across the 12 pitch classes (semitones)**
- Useful for analyzing the **harmonic** content of audio

### Key points - Examine spectrum of <font color="#DC143C">"synthetic voice" vs "original voice"</font>
- Chroma features compress an audio signal into 12 bins, one for each semitone in an octave, focusing on pitch class rather than exact frequency.  
- They capture harmonic structure, which helps distinguish different sounds (like different phonemes in speech) because each has a characteristic pattern of harmonics.  
- This makes them valuable for tasks like speech recognition and transcription, where recognizing subtle differences between sounds is important.  
- Librosa’s `chroma_stft` function in Python makes it easy to compute these features from an audio signal.  
- In systems like Whisper, adding chroma features to the model’s inputs can improve robustness, especially in noisy or multi-speaker environments, by giving the model more detailed harmonic information to work with.

### Chroma Features: <font color="#DC143C">TTS Voice (1st plot) versus Original Human Voice (2nd plot)</font>

In [ ]:
%matplotlib inline

import librosa
import librosa.display
import matplotlib.pyplot as plt

# 1. Load an audio file (mono)
audio_path = "ZD_AE_tts.wav"
# audio_path = "output_cloned/test_sample_03.wav"
mono_signal, sample_rate = librosa.load(audio_path, sr=None, mono=True)

# 2. Compute the chroma features
chroma_features = librosa.feature.chroma_stft(y=mono_signal, sr=sample_rate)

# 3. Plot the chroma features
plt.figure(figsize=(10, 4))
librosa.display.specshow(
    chroma_features,
    sr=sample_rate,
    hop_length=512,
    x_axis="time",
    y_axis="chroma"
)
plt.title("Synthetic TTS Voice Chroma features")
plt.colorbar(format="%+0.2f")
plt.tight_layout()
plt.show()


In [ ]:
%matplotlib inline

import librosa
import librosa.display
import matplotlib.pyplot as plt

# 1. Load an audio file (mono)
# Replace "example.wav" with your audio path
audio_path = "voices/zd/1.wav"    #"voices/zd/1.wav"
mono_signal, sample_rate = librosa.load(audio_path, sr=None, mono=True)

# 2. Compute the chroma features
chroma_features = librosa.feature.chroma_stft(y=mono_signal, sr=sample_rate)

# 3. Plot the chroma features
plt.figure(figsize=(10, 4))
librosa.display.specshow(
    chroma_features,
    sr=sample_rate,
    hop_length=512,
    x_axis="time",
    y_axis="chroma"
)
plt.title("Original Human Voice Chroma features")
plt.colorbar(format="%+0.2f")
plt.tight_layout()
plt.show()